# TD3 Agent Testing Guide on Pendulum-v1

- Twin Delayed Deep Deterministic policy gradient (TD3) is an off-policy, actor-critic algorithm designed for continuous action spaces, addressing the overestimation bias inherent in standard actor-critic algorithms like DDPG.
- While DDPG often suffers from function approximation errors that cause the critic to overestimate Q-values — which in turn misleads the deterministic policy—TD3 introduces three primary improvements:
  1. **Clipped Double Q-learning (Twin Critics)**: Trains two independent critics ($Q_1, Q_2$) and uses the minimum value estimate when computing the Bellman target ($y = r + \gamma \min_{i=1,2} Q_i(s', a')$), preventing overestimation bias.
  2. **Target Policy Smoothing**: Adds clipped Gaussian noise to the target action ($a' = \text{clip}(\mu_{\text{targ}}(s') + \epsilon, -a_{\max}, a_{\max})$ where $\epsilon \sim \text{clip}(\mathcal{N}(0, \sigma), -c, c)$), smoothing the value landscape and preventing exploitation of sharp peaks in Q-function estimates.
  3. **Delayed Policy Updates**: Updates the actor network and target networks less frequently than the critics (typically once every 2 critic updates), allowing value estimates to stabilize before the policy is updated.

This notebook provides a minimal workflow for testing a Twin Delayed Deep Deterministic Policy Gradient (TD3) agent on `Pendulum-v1`. It configures the experiment, trains the agent, evaluates the deterministic policy, and prints the metrics returned by `learn()`. The cells are intended to be edited and rerun while experimenting with the agent and its hyperparameters.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import gymnasium as gym
import numpy as np

POLICY_GRADIENT_DIR = Path.cwd()
if POLICY_GRADIENT_DIR.name != 'policy-gradient':
    POLICY_GRADIENT_DIR = Path('/home/dangineer/Learnspaces/coursera/reinforcement-learning-specialization/rl-lab/deep-rl/policy-gradient')
if str(POLICY_GRADIENT_DIR) not in sys.path:
    sys.path.insert(0, str(POLICY_GRADIENT_DIR))

DEEP_RL_DIR = POLICY_GRADIENT_DIR.parent
if str(DEEP_RL_DIR) not in sys.path:
    sys.path.insert(0, str(DEEP_RL_DIR))

from td3 import TD3


## 1. Configure the Experiment

Adjust these values before running the training cell. Start with a small `MAX_STEPS` while debugging, then increase it for longer experiments.

In [3]:
ENV_ID = 'Pendulum-v1'
SEED = 2026
MAX_STEPS = 20_000
START_STEPS = 1_000
BUFFER_SIZE = 100_000
BATCH_SIZE = 64
ACTOR_HIDDEN_DIMS = (256, 256)
CRITIC_HIDDEN_DIMS = (256, 256)
ACTOR_LR = 1e-4
CRITIC_LR = 1e-3
GAMMA = 0.99
TAU = 0.005
EXPLORATION_NOISE = 0.1
TARGET_POLICY_NOISE = 0.2
TARGET_NOISE_CLIP = 0.5

print(f'Environment: {ENV_ID}')
print(f'Seed: {SEED}')
print(f'Total steps: {MAX_STEPS}')
print(f'Warmup (start) steps: {START_STEPS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Soft update tau: {TAU}')
print(f'Exploration noise: {EXPLORATION_NOISE}')
print(f'Target policy noise: {TARGET_POLICY_NOISE}')
print(f'Target noise clip: {TARGET_NOISE_CLIP}')


Environment: Pendulum-v1
Seed: 2026
Total steps: 20000
Warmup (start) steps: 1000
Batch size: 64
Soft update tau: 0.005
Exploration noise: 0.1
Target policy noise: 0.2
Target noise clip: 0.5


## 2. Create the Environment and Test the Agent

This cell creates a single continuous environment, initializes TD3, runs training with `learn()`, tests the learned deterministic policy with `evaluate()`, and prints the returned metrics. Rerun this cell after changing the configuration above.

In [4]:
env = gym.make(ENV_ID)
env.reset(seed=SEED)
env.action_space.seed(SEED)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])

agent = TD3(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    actor_hidden_dims=ACTOR_HIDDEN_DIMS,
    critic_hidden_dims=CRITIC_HIDDEN_DIMS,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    gamma=GAMMA,
    tau=TAU,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    start_steps=START_STEPS,
    exploration_noise=EXPLORATION_NOISE,
    target_policy_noise=TARGET_POLICY_NOISE,
    target_noise_clip=TARGET_NOISE_CLIP,
    seed=SEED,
)

print(f'State dim: {state_dim}, Action dim: {action_dim}, Max action: {max_action}')
print(f'Starting training for {MAX_STEPS} steps...')
metrics = agent.learn(env, max_steps=MAX_STEPS)

print('\nEvaluating trained agent (without exploration noise)...')
eval_score = agent.evaluate(env, num_episodes=5)
print(f'Evaluation Mean Return over 5 episodes: {eval_score:.2f}')
env.close()

print('\nTraining metrics:')
for name, value in metrics.items():
    print(f'  {name}: {value}')


State dim: 3, Action dim: 1, Max action: 2.0
Starting training for 20000 steps...



Evaluating trained agent (without exploration noise)...
Evaluation Mean Return over 5 episodes: -163.95

Training metrics:
  actor_loss: 55.27031136700862
  critic_loss: 1.500912314963638
  critic2_loss: 1.3479698166506744
  q_value: -56.565367571381
  q2_value: -56.56461185357908
  episode_return: -466.07354764209356
  episode_length: 200.0
  total_steps: 20000


## 3. Inspect the Training Metrics

`TD3.learn()` returns aggregate information from network updates and completed training episodes. The next cell prints a short explanation for each metric.

In [5]:
metric_descriptions = {
    'actor_loss': 'Mean negative Q1-value objective (-Q1(s, mu(s))). As policy improves, this tends to decrease.',
    'critic_loss': 'Critic 1 Bellman mean squared error against the clipped double Q-target.',
    'critic2_loss': 'Critic 2 Bellman mean squared error against the clipped double Q-target.',
    'q_value': 'Average estimated Q1-value over the sampled training mini-batches.',
    'q2_value': 'Average estimated Q2-value over the sampled training mini-batches.',
    'episode_return': 'Average episode return during training with exploration noise.',
    'episode_length': 'Average episode duration in steps.',
    'total_steps': 'Total environment interaction steps taken.',
}

for name, description in metric_descriptions.items():
    print(f'{name}: {metrics.get(name, 0.0)}')
    print(f'  {description}')


actor_loss: 55.27031136700862
  Mean negative Q1-value objective (-Q1(s, mu(s))). As policy improves, this tends to decrease.
critic_loss: 1.500912314963638
  Critic 1 Bellman mean squared error against the clipped double Q-target.
critic2_loss: 1.3479698166506744
  Critic 2 Bellman mean squared error against the clipped double Q-target.
q_value: -56.565367571381
  Average estimated Q1-value over the sampled training mini-batches.
q2_value: -56.56461185357908
  Average estimated Q2-value over the sampled training mini-batches.
episode_return: -466.07354764209356
  Average episode return during training with exploration noise.
episode_length: 200.0
  Average episode duration in steps.
total_steps: 20000
  Total environment interaction steps taken.


## 4. Hyperparameter Tuning

TD3 improves on DDPG with three key techniques that reduce overestimation bias and training variance:

- **Twin critics (Clipped Double Q-learning)**: two independent critics are trained, and the smaller of their target predictions is used: $y = r + \gamma (1 - d) \min(Q_{\text{targ1}}, Q_{\text{targ2}})$.
- **Target policy smoothing**: clipped Gaussian noise is added to the target action so that Q-values stay smooth across neighboring actions. Standard values: `target_policy_noise = 0.2`, `target_noise_clip = 0.5`.
- **Delayed policy updates**: the actor and target networks are updated less often than the critics, with `policy_delay = 2`.

**Configuration:**

- **Learning rates**: `critic_lr = 1e-3`, `actor_lr = 1e-4` to `3e-4`. The critic's rate is higher so the critics learn faster than the policy changes.
- **Network**: MLP `(256, 256)` with ReLU, sufficient for benchmarks such as Pendulum.
- **Soft update rate**: `tau = 0.005`, so each update blends 0.5% of the new weights into the target networks.
- **Exploration noise**: `exploration_noise = 0.1 * max_action`, Gaussian noise added to actions during training.
- **Warm-up**: `start_steps = 1_000`, which seeds the replay buffer with diverse initial transitions.